**when / otherwise — Conditional Logic**

when() and otherwise() are used to implement conditional logic in PySpark DataFrames. They work like IF...ELSE statements in programming languages or CASE WHEN in SQL

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-9")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ff649077-f66e-429b-b537-36b730f47336;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 138ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

Add a column order_size to orders.csv using when(): if quantity >= 4 then "Large", if >= 2 then "Medium", otherwise "Small". Show order_id, quantity, and order_size.

In [2]:
from pyspark.sql import functions as F
orders_df=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv",
header=True,
inferSchema=True

)
orders_df.withColumn(
    "order_size",
    F.when(F.col("quantity") >= 4, "large")
     .when(F.col("quantity") >= 2, "medium")
     .otherwise("small")
).select(
    "order_id",
    "quantity",
    "order_size"
).show(10)

26/08/09 07:38:46 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------+--------+----------+
|order_id|quantity|order_size|
+--------+--------+----------+
|   O0001|       2|    medium|
|   O0002|       1|     small|
|   O0003|       4|     large|
|   O0004|       2|    medium|
|   O0005|       3|    medium|
|   O0006|       1|     small|
|   O0007|       2|    medium|
|   O0008|       1|     small|
|   O0009|       1|     small|
|   O0010|       2|    medium|
+--------+--------+----------+
only showing top 10 rows


**Task 2**

Add a column discount_label: if discount_pct == 0 then "No Discount", if <= 10 then "Low Discount", if <= 15 then "Medium Discount", otherwise "High Discount". Show the distribution using groupBy("discount_label").count().

In [3]:
from pyspark.sql import functions as F

orders_df.withColumn(
    "discount_label",
    F.when(F.col("discount_pct") == 0, "No Discount")
     .when(F.col("discount_pct") <= 10, "Low Discount")
     .when(F.col("discount_pct") <= 15, "Medium Discount")
     .otherwise("High Discount")
).groupBy("discount_label").count().show()

+---------------+-----+
| discount_label|count|
+---------------+-----+
|Medium Discount|    8|
|    No Discount|   54|
|   Low Discount|   33|
|  High Discount|    5|
+---------------+-----+



**Task 3**

Using compound conditions, add a column vip_order: true when unit_price > 500 AND status == "Delivered", false otherwise. How many VIP orders are there?

In [4]:
Vip_order_count=orders_df.withColumn("vip_order",
F.when((F.col("unit_price")>500) & (F.col("status")=="Delivered"),"true").\
    otherwise("false")).filter(F.col('vip_order')=='true').count()
print(f"Total number of vip orders :{Vip_order_count}")

Total number of vip orders :15


**Task 4**

Add a regional_price column where East region gets 10% off, West gets 15% off, and all other regions pay full price. Use when() returning a column expression. Show order_id, region, unit_price, and regional_price.

In [5]:
from pyspark.sql import functions as F

orders_df = orders_df.withColumn(
    "regional_price",
    F.when(
        F.col("region") == "East",
        F.round(F.col("unit_price") * 0.90,2)
    ).when(
        F.col("region") == "West",
        F.round(F.col("unit_price") * 0.85, 2)
    ).otherwise(
        F.col("unit_price")
    )
)

orders_df.select(
    "order_id",
    "region",
    "unit_price",
    "regional_price"
).show()

+--------+-------+----------+--------------+
|order_id| region|unit_price|regional_price|
+--------+-------+----------+--------------+
|   O0001|   East|   1299.99|       1169.99|
|   O0002|   West|    449.99|        382.49|
|   O0003|Midwest|    349.99|        349.99|
|   O0004|  South|     89.99|         89.99|
|   O0005|   West|     29.99|         25.49|
|   O0006|   East|    199.99|        179.99|
|   O0007|  South|    109.99|        109.99|
|   O0008|   West|     59.99|         50.99|
|   O0009|  South|   1299.99|       1299.99|
|   O0010|   West|     79.99|         67.99|
|   O0011|  South|     39.99|         39.99|
|   O0012|   East|    599.99|        539.99|
|   O0013|   West|     49.99|         42.49|
|   O0014|Midwest|     89.99|         89.99|
|   O0015|   East|    699.99|        629.99|
|   O0016|Midwest|     29.99|         29.99|
|   O0017|   West|    199.99|        169.99|
|   O0018|   West|     24.99|         21.24|
|   O0019|  South|    129.99|        129.99|
|   O0020|

In [6]:
spark.stop()